# 🌐 Lendo um CSV Diretamente da Internet

![Jupyter](https://img.shields.io/badge/Jupyter-111827?style=flat-square&logo=jupyter&logoColor=F37626)
![Python](https://img.shields.io/badge/Python-111827?style=flat-square&logo=python&logoColor=3776AB)
![Pandas](https://img.shields.io/badge/Pandas-150458?style=flat-square&logo=pandas&logoColor=white)
![Python Version](https://img.shields.io/badge/python-3.14+-blue)
![Tópico](https://img.shields.io/badge/tópico-leitura%20de%20dados%20%7C%20web-teal)
![Dificuldade](https://img.shields.io/badge/dificuldade-Intermediário-yellow)
![Pré-req](https://img.shields.io/badge/pré--req-dataframe%20%7C%20read__csv-purple)
![Biblioteca](https://img.shields.io/badge/requer-pip%20install%20pandas%20requests-orange)

> Nem sempre nossos dados estão salvos em um arquivo local — às vezes eles estão disponíveis direto na internet. O pandas consegue ler um csv direto de um link, mas nem sempre é tão simples assim.


## 📋 Conteúdo

1. [Introdução](#-1-introdução)
2. [Caso 1: csv direto no link](#-2-caso-1-csv-direto-no-link)
3. [Caso 2: csv dentro de uma requisição que precisa ser tratada](#-3-caso-2-csv-dentro-de-uma-requisição-que-precisa-ser-tratada)
4. [Lembrete: principais tipos de encoding](#-4-lembrete-principais-tipos-de-encoding)

---

## 🌍 1. Introdução

Essa é uma das formas de importar arquivos da internet, direto pelo pandas. Mas nem sempre você vai conseguir abrir assim direto.

Existem 2 situações principais em que você consegue trazer os dados para um DataFrame:

1. O arquivo csv está direto no link (melhor dos mundos).
2. Os dados existem, mas ficam no meio de uma página/requisição que precisa ser tratada antes de virar um DataFrame.

## 🔗 2. Caso 1: csv direto no link

Quando o link aponta direto para um arquivo `.csv`, o `pd.read_csv()` funciona exatamente como se estivéssemos lendo um arquivo local — só que passamos a URL no lugar do caminho do arquivo.

```python
dataframe = pd.read_csv(url)
```

Disponibilizei um arquivo csv para download no Google Drive, com o link de exportação direta:

https://drive.google.com/uc?authuser=0&id=1Ru7s-x3YJuStZK1mqr_qNqiHVvdHUN66&export=download

In [1]:
import pandas as pd

url_drive = 'https://drive.google.com/uc?authuser=0&id=1Ru7s-x3YJuStZK1mqr_qNqiHVvdHUN66&export=download'

cotacao_dolar_df = pd.read_csv(url_drive)
display(cotacao_dolar_df.head())

,Date,Price,Open,High,Low,Change %
0,"Jan 01, 2020",4.0195,4.0195,4.0195,4.0195,0.00%
1,"Dec 31, 2019",4.0195,4.0201,4.0201,4.0195,0.00%
2,"Dec 30, 2019",4.0195,4.0484,4.0484,4.0084,-0.65%
3,"Dec 27, 2019",4.0460,4.0568,4.0614,4.0339,-0.25%
4,"Dec 26, 2019",4.0560,4.0827,4.0827,4.0453,-0.68%


## 🧵 3. Caso 2: csv dentro de uma requisição que precisa ser tratada

Pesquisei por histórico de preços do café no Google e cheguei nesse site:

https://portalweb.cooxupe.com.br:9080/portal/precohistoricocafe_2.jsp

Só que aqui não existe nenhum arquivo `.csv` para baixar — o link é de uma página HTML, e os dados que queremos estão dentro de uma **tabela** no meio dessa página. Nesse caso o `pd.read_csv()` não resolve sozinho.

O caminho é:

1. Usar a biblioteca `requests` para baixar o conteúdo (HTML) da página.
2. Usar `pd.read_html()` para procurar tabelas dentro desse HTML e transformá-las em DataFrame(s).

```python
resposta = requests.get(url)
tabelas = pd.read_html(resposta.text)
```

| Parâmetro | O que faz |
|-----------|-----------|
| `resposta.text` | Texto (HTML) retornado pela requisição |
| `tabelas` | Lista com **todos** os DataFrames encontrados na página — cada `<table>` vira um item da lista |

> 💡 **Dica:** depois de montado, esse DataFrame pode ser exportado para `.csv` normalmente com `.to_csv()`, assim como já fizemos antes.

In [2]:
import io
import requests

url_cafe = 'https://portalweb.cooxupe.com.br:9080/portal/precohistoricocafe_2.jsp'

resposta = requests.get(url_cafe)
tabelas = pd.read_html(io.StringIO(resposta.text))

print(f'Foram encontradas {len(tabelas)} tabela(s) na página')

preco_cafe_df = tabelas[0]
display(preco_cafe_df.head())

Foram encontradas 1 tabela(s) na página


,Ano,Janeiro,Fevereiro,Março,Abril,Maio,Junho,Julho,Agosto,Setembro,Outubro,Novembro,Dezembro
0,2025,2256.79,2564.10,2554.43,2564.42,2382.57,2152.81,NaN,NaN,NaN,NaN,NaN,NaN
1,2024,974.46,1005.54,1023.45,1145.12,1186.17,1240.85,1216.35,1279.77,1316.36,1463.25,1732.67,2063.69
2,2023,1030.14,1130.26,1103.37,1115.24,1030.64,927.88,878.50,868.24,802.59,800.95,845.60,925.46
3,2022,1480.49,1488.11,1258.22,1253.25,1259.21,1256.28,1119.45,1075.91,1098.56,1032.75,915.26,994.22
4,2021,639.23,695.33,726.88,744.05,811.06,793.67,762.36,772.03,851.11,1113.29,1285.94,1446.66


## 🔤 4. Lembrete: principais tipos de encoding

Sempre que um `read_csv` (ou `read_html`) vier com acentos/caracteres estranhos, o problema costuma ser o `encoding`. Vale lembrar os principais que vamos usar ao longo do curso:

- `encoding='latin1'`
- `encoding='ISO-8859-1'`
- `encoding='utf-8'`
- `encoding='cp1252'`

> ⚠️ **Cuidado:** nem todo site informa o encoding certo de cara — às vezes é necessário testar mais de uma opção até os caracteres aparecerem corretamente.